<a href="https://colab.research.google.com/github/ReffkiAndreaPratama/aksara-detect/blob/main/colab_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AksaraDetect - Full Project di Google Colab

**Sebelum mulai:**
1. Klik **Runtime** -> **Change runtime type** -> pilih **T4 GPU** -> Save
2. Ganti `GANTI_API_KEY_KAMU` di Cell 2 dengan API key Roboflow kamu
3. Klik **Runtime** -> **Run all**
4. Tunggu ~15 menit, lalu klik URL yang muncul di Cell terakhir

**Cara dapat API Key:** buka https://app.roboflow.com -> klik foto profil -> Settings -> Roboflow API

In [5]:
# Cell 1 - Install semua dependensi
!nvidia-smi
!pip install ultralytics roboflow streamlit pyngrok PyYAML matplotlib seaborn pandas Pillow scikit-learn -q
print("Install selesai!")

Mon May 11 06:19:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# Cell 2 - Download dataset dari Roboflow
from roboflow import Roboflow
import yaml, os

API_KEY = "cSaQdx9RY5YTrc7KjCAs"  # <-- GANTI INI
# Cara dapat: buka app.roboflow.com -> Settings -> Roboflow API

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("novalrizkiansyah-ymail-com").project("aksara-ulu-rejang")
dataset = project.version(4).download("yolov8")

DATA_YAML = dataset.location + "/data.yaml"
base = os.path.dirname(DATA_YAML)

with open(DATA_YAML) as f:
    data = yaml.safe_load(f)

data["train"] = base + "/train/images"
data["val"]   = base + "/valid/images"
data["test"]  = base + "/test/images"

with open(DATA_YAML, 'w') as f:
    yaml.dump(data, f)

print(f"Dataset siap! Kelas: {data['nc']}")

loading Roboflow workspace...
loading Roboflow project...
Dataset siap! Kelas: 253


In [7]:
# Cell 3 - Clone project dari GitHub
!git clone https://github.com/ReffkiAndreaPratama/aksara-detect.git /content/app
print("Project berhasil di-clone!")

Cloning into '/content/app'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 45 (delta 9), reused 32 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 66.09 KiB | 1.02 MiB/s, done.
Resolving deltas: 100% (9/9), done.
Project berhasil di-clone!


In [8]:
# Cell 4 - Training YOLOv8 dengan GPU (~10-15 menit)
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data     = DATA_YAML,
    epochs   = 50,
    imgsz    = 640,
    batch    = 32,
    device   = 0,
    project  = "/content/runs",
    name     = "aksara_ulu",
    exist_ok = True,
)
print("Training selesai!")
print("mAP50:", results.results_dict.get("metrics/mAP50(B)", "N/A"))

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Aksara-Ulu-Rejang-4/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=aksara_ulu, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pat

In [54]:
# Cell 5 - Setup model, label map, dan config
import os, json, shutil, yaml

for d in ["/content/app/models", "/content/app/results", "/content/app/logs"]:
    os.makedirs(d, exist_ok=True)

shutil.copy("/content/runs/aksara_ulu/weights/best.pt", "/content/app/models/best.pt")

try:
    with open("/content/app/models/metrics.json", "w") as f:
        json.dump(results.results_dict, f, indent=2)
    print("Metrics saved successfully from training results.")
except NameError:
    print("Error: 'results' object not found. Please ensure Cell 4 (training) was run successfully before running Cell 5.")
except Exception as e:
    print(f"An unexpected error occurred while saving metrics: {e}")

with open(DATA_YAML) as f:
    d = yaml.safe_load(f)
label_map = {str(i): name for i, name in enumerate(d["names"])}
with open("/content/app/models/label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

config_lines = [
    "import os\n",
    'BASE_DIR        = "/content/app"\n',
    f'DATA_YAML       = "{DATA_YAML}"\n',
    'MODEL_DIR       = "/content/app/models"\n',
    'RESULT_DIR      = "/content/app/results"\n',
    'LOG_DIR         = "/content/app/logs"\n',
    'RUNS_DIR        = "/content/app/runs"\n',
    'MODEL_BEST_PATH = "/content/app/models/best.pt"\n',
    'MODEL_LAST_PATH = "/content/app/models/last.pt"\n',
    'LABEL_MAP_PATH  = "/content/app/models/label_map.json"\n',
    'METRICS_PATH    = "/content/app/models/metrics.json"\n',
    'PREDICT_LOG     = "/content/app/logs/prediction_log.json"\n',
    "CONF_THRESHOLD  = 0.25\n",
    "IOU_THRESHOLD   = 0.45\n",
    'APP_NAME        = "AksaraDetect"\n',
    'APP_VERSION     = "1.0.0"\n',
    'APP_TAGLINE     = "Aksara Ulu Rejang - YOLOv8"\n',
    'MODEL_SIZE      = "n"\n',
    'EPOCHS          = 50\n',
    'IMG_SIZE        = 640\n',
    'BATCH_SIZE      = 32\n',
    'PATIENCE        = 100\n',
    'LR0             = 0.01\n',
    'DEVICE          = "0"\n',
    'WORKERS         = 8\n',
    'SEED            = 0\n',
    'LOGS_DIR        = "/content/app/logs"\n',
]
with open("/content/app/src/config.py", "w") as f:
    f.writelines(config_lines)

print(f"Setup selesai! {len(label_map)} kelas siap.")

Metrics saved successfully from training results.
Setup selesai! 253 kelas siap.


In [64]:
#cell 6
import subprocess
import time
import sys

from pyngrok import ngrok

# =========================
# TUTUP SEMUA SESSION LAMA
# =========================
ngrok.kill()

# =========================
# AUTH TOKEN
# =========================
ngrok.set_auth_token("3DZO90kEo4F5COhRtjSMauVV9Pi_2yCBGomGodEXAN3BPmWuJ")

# =========================
# JALANKAN STREAMLIT
# =========================
proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "/content/app/src/app.py",
        "--server.port",
        "8501",
        "--server.headless",
        "true",
        "--server.enableCORS",
        "false",
        "--server.enableXsrfProtection",
        "false",
    ],
    cwd="/content/app/src"
)

time.sleep(10)

# =========================
# BUAT TUNNEL BARU
# =========================
tunnel = ngrok.connect(8501)

print("=" * 55)
print("AKSARADETECT SIAP!")
print("Buka URL ini:")
print(tunnel.public_url)
print("=" * 55)

AKSARADETECT SIAP!
Buka URL ini:
https://storage-evacuee-skeptic.ngrok-free.dev
